# 🌿 VedaVision — Classification: Tune and Evaluate Every Model

This notebook trains, hyperparameter-tunes, refits, and evaluates all six candidate classifiers:

1. Logistic Regression
2. K-Nearest Neighbors
3. Random Forest
4. XGBoost
5. SVM with RBF kernel
6. Gradient Boosting

## What was fixed

- Replaced the very expensive exhaustive grid with **bounded `RandomizedSearchCV`** for every model.
- Added **early stopping and a smaller search budget for Gradient Boosting**.
- Uses **three-fold tuning by default** and allows `quick`, `balanced`, or `thorough` profiles.
- Moves imputation and scaling into a **scikit-learn Pipeline**, preventing preprocessing leakage between CV folds.
- Uses `probability=False` for SVM during training because probability calibration was unused and slow.
- Saves a **checkpoint after each model**, so a Colab interruption does not lose completed searches.
- Selects the deployment model using **cross-validation performance before inspecting test scores**.
- Refits every tuned model on the full training dataset and evaluates every model on the held-out test dataset.
- Produces accuracy, balanced accuracy, macro precision/recall/F1, weighted F1, classification reports, confusion matrices, predictions, timing information, and saved model pipelines.

> Default profile: `balanced`. It is designed to complete in Colab while still tuning every model. Change it to `thorough` only when you have a long-running/high-RAM runtime.

## 1. Install libraries

In [ ]:
!pip install -q scikit-learn xgboost joblib matplotlib seaborn pandas numpy

import sklearn
import xgboost
print('✅ Libraries installed')
print('scikit-learn:', sklearn.__version__)
print('xgboost     :', xgboost.__version__)

✅ Libraries installed
scikit-learn: 1.9.0
xgboost     : 3.3.0


## 2. Imports

In [ ]:
import os
import re
import json
import time
import hashlib
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    StratifiedGroupKFold,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
warnings.filterwarnings('ignore', category=UserWarning)
np.random.seed(42)

## 3. Mount Drive and configure the run

Only this cell normally needs editing.

### Tuning profiles

- `quick`: confirms that the full notebook works; smallest search.
- `balanced`: recommended default; tunes every model with a practical runtime.
- `thorough`: larger searches and five-fold CV; substantially slower.

### Group-aware splitting

When augmented images or top/bottom views can come from the same physical leaf, add a source identifier to the CSV and set `GROUP_COLUMN` to that column. All related images will then stay in the same partition/fold.

In [ ]:
 
# ============================================================
# PATHS
# ============================================================
CSV_PATH   = 'C:/Users/User/Documents/UOM/L4S2/Research/csv'
MODEL_PATH = 'C:/Users/User/Documents/UOM/L4S2/Research'
LOGS_PATH  = 'C:/Users/User/Documents/UOM/L4S2/Research/csv/logs'

TRAIN_CSV       = os.path.join(CSV_PATH, 'species_id_train_features.csv')
TEST_CSV        = os.path.join(CSV_PATH, 'species_id_test_features.csv')
TEST_LABELS_CSV = os.path.join(CSV_PATH, 'species_id_test_labels.csv')

# ============================================================
# RUN CONFIGURATION
# ============================================================
RANDOM_STATE = 42
VAL_SIZE = 0.20
TUNING_PROFILE = 'balanced'       # 'quick', 'balanced', or 'thorough'
SELECTION_METRIC = 'f1_macro'     # used by CV and model selection
SEARCH_N_JOBS = 2                 # safer than -1 on Colab; XGBoost itself uses 1 thread
RESUME_FROM_CHECKPOINTS = True
FORCE_RETRAIN_MODELS = []         # e.g. ['Gradient Boosting']

# Set this when the CSV contains an original-image/physical-leaf identifier.
# Example: GROUP_COLUMN = 'source_leaf_id'
GROUP_COLUMN = None

# Optional: run a subset while debugging. Keep None for the full requested run.
RUN_ONLY_MODELS = None             # e.g. ['Gradient Boosting']

PROFILE_SETTINGS = {
    'quick': {
        'cv_splits': 3,
        'iterations': {
            'Logistic Regression': 4,
            'K-Nearest Neighbors': 5,
            'Random Forest': 6,
            'XGBoost': 6,
            'SVM (RBF)': 5,
            'Gradient Boosting': 5,
        },
    },
    'balanced': {
        'cv_splits': 3,
        'iterations': {
            'Logistic Regression': 8,
            'K-Nearest Neighbors': 10,
            'Random Forest': 12,
            'XGBoost': 14,
            'SVM (RBF)': 10,
            'Gradient Boosting': 8,
        },
    },
    'thorough': {
        'cv_splits': 5,
        'iterations': {
            'Logistic Regression': 12,
            'K-Nearest Neighbors': 16,
            'Random Forest': 24,
            'XGBoost': 28,
            'SVM (RBF)': 18,
            'Gradient Boosting': 16,
        },
    },
}

if TUNING_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f'Unknown TUNING_PROFILE: {TUNING_PROFILE}')

CV_SPLITS = PROFILE_SETTINGS[TUNING_PROFILE]['cv_splits']
SEARCH_ITERATIONS = PROFILE_SETTINGS[TUNING_PROFILE]['iterations']

CHECKPOINT_PATH = os.path.join(MODEL_PATH, 'search_checkpoints')
ALL_MODELS_PATH = os.path.join(MODEL_PATH, 'all_models')
REPORTS_PATH = os.path.join(LOGS_PATH, '05_model_reports')

for path in [MODEL_PATH, LOGS_PATH, CHECKPOINT_PATH, ALL_MODELS_PATH, REPORTS_PATH]:
    os.makedirs(path, exist_ok=True)

print('✅ Configuration ready')
print('Profile       :', TUNING_PROFILE)
print('CV folds      :', CV_SPLITS)
print('Scoring       :', SELECTION_METRIC)
print('Search workers:', SEARCH_N_JOBS)
print('Checkpoints   :', CHECKPOINT_PATH)

✅ Configuration ready
Profile       : balanced
CV folds      : 3
Scoring       : f1_macro
Search workers: 2
Checkpoints   : C:/Users/User/Documents/UOM/L4S2/Research\search_checkpoints


## 4. Load training features, test features, and test labels

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
test_labels_df = pd.read_csv(TEST_LABELS_CSV)

print('📊 TRAIN features:', train_df.shape)
print('📊 TEST features :', test_df.shape)
print('📊 TEST labels   :', test_labels_df.shape)
print()
print('Species in training data:')
print(sorted(train_df['species'].astype(str).unique()))
print()
print('Training samples per species/view:')
print(train_df.groupby(['species', 'view']).size().to_string())

📊 TRAIN features: (6995, 109)
📊 TEST features : (130, 107)
📊 TEST labels   : (130, 3)

Species in training data:
['Diya_Na', 'Gammiris', 'Ingini', 'Iriveriya', 'Kapparawalliya', 'Kora_Kaha', 'Kuringchan', 'Kurundu', 'Masbadda', 'Na', 'Rathu_Koboleela', 'Sudu_Koboleela', 'Wali_Kaha']

Training samples per species/view:
species          view  
Diya_Na          bottom    260
                 top       264
Gammiris         bottom    270
                 top       270
Ingini           bottom    270
                 top       270
Iriveriya        bottom    270
                 top       270
Kapparawalliya   bottom    270
                 top       270
Kora_Kaha        bottom    270
                 top       270
Kuringchan       bottom    270
                 top       270
Kurundu          bottom    270
                 top       270
Masbadda         bottom    270
                 top       270
Na               bottom    270
                 top       270
Rathu_Koboleela  bottom    270
     

## 5. Validate files and prevent silent data mistakes

In [ ]:
REQUIRED_TRAIN_COLUMNS = {'filename', 'species', 'view', 'label'}
REQUIRED_TEST_COLUMNS = {'filename', 'view'}
REQUIRED_TEST_LABEL_COLUMNS = {'filename', 'view', 'true_label'}


def require_columns(df, required, name):
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}')


require_columns(train_df, REQUIRED_TRAIN_COLUMNS, 'train_df')
require_columns(test_df, REQUIRED_TEST_COLUMNS, 'test_df')
require_columns(test_labels_df, REQUIRED_TEST_LABEL_COLUMNS, 'test_labels_df')

# Confirm that the two training label columns agree.
label_mismatches = train_df[
    train_df['species'].astype(str) != train_df['label'].astype(str)
]
if not label_mismatches.empty:
    raise ValueError(
        f'{len(label_mismatches)} training rows have different species and label values.'
    )


def duplicate_count(df, columns):
    return int(df.duplicated(subset=columns).sum())

checks = {
    'train duplicate filename/view rows': duplicate_count(train_df, ['filename', 'view']),
    'test duplicate filename/view rows': duplicate_count(test_df, ['filename', 'view']),
    'test-label duplicate filename/view rows': duplicate_count(test_labels_df, ['filename', 'view']),
}

for name, value in checks.items():
    print(f'{name:<42}: {value}')

if any(value > 0 for value in checks.values()):
    raise ValueError('Duplicate filename/view rows were found. Fix them before training.')

# Left merge while preserving exact test-row order.
test_keys = test_df[['filename', 'view']].copy()
test_keys['_row_order'] = np.arange(len(test_keys))
test_with_labels = test_keys.merge(
    test_labels_df[['filename', 'view', 'true_label']],
    on=['filename', 'view'],
    how='left',
    validate='one_to_one',
).sort_values('_row_order').reset_index(drop=True)

if len(test_with_labels) != len(test_df):
    raise ValueError('Test-label merge changed row count.')
if test_with_labels['true_label'].isna().any():
    raise ValueError('One or more test rows do not have a matching true label.')

exact_filename_overlap = set(train_df['filename']) & set(test_df['filename'])
print('Exact train/test filename overlap:', len(exact_filename_overlap))
if exact_filename_overlap:
    raise ValueError('Exact filenames appear in both train and test data.')

if GROUP_COLUMN is not None:
    if GROUP_COLUMN not in train_df.columns:
        raise ValueError(f'GROUP_COLUMN={GROUP_COLUMN!r} is not in train_df.')
    print(f'✅ Group-aware splitting enabled with {GROUP_COLUMN!r}')
else:
    print('⚠️ GROUP_COLUMN is not configured.')
    print('   Exact filename leakage is absent, but augmented siblings or paired')
    print('   views cannot be detected without an original-image/leaf identifier.')

print('✅ Structural data checks passed')

train duplicate filename/view rows        : 0
test duplicate filename/view rows         : 0
test-label duplicate filename/view rows   : 0
Exact train/test filename overlap: 0
⚠️ GROUP_COLUMN is not configured.
   Exact filename leakage is absent, but augmented siblings or paired
   views cannot be detected without an original-image/leaf identifier.
✅ Structural data checks passed


## 6. Prepare model features and encode labels

In [ ]:
META_COLS = ['filename', 'species', 'view', 'label']
if GROUP_COLUMN is not None:
    META_COLS.append(GROUP_COLUMN)

feature_cols = [column for column in train_df.columns if column not in META_COLS]

# Test must contain every training feature. Extra test metadata is allowed.
missing_test_features = sorted(set(feature_cols) - set(test_df.columns))
if missing_test_features:
    raise ValueError(f'Test CSV is missing feature columns: {missing_test_features}')

X = train_df[feature_cols].copy()
X['view_top'] = train_df['view'].astype(str).str.lower().eq('top').astype(int)
feature_cols_final = list(X.columns)

X_test = test_df[feature_cols].copy()
X_test['view_top'] = test_df['view'].astype(str).str.lower().eq('top').astype(int)
X_test = X_test[feature_cols_final]

# Convert all model inputs to numeric. Unexpected strings become NaN and are imputed inside CV.
X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_test = X_test.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(train_df['label'].astype(str))

unknown_test_labels = sorted(
    set(test_with_labels['true_label'].astype(str)) - set(label_encoder.classes_)
)
if unknown_test_labels:
    raise ValueError(f'Test labels not seen during training: {unknown_test_labels}')

y_test_encoded = label_encoder.transform(test_with_labels['true_label'].astype(str))

groups_all = None if GROUP_COLUMN is None else train_df[GROUP_COLUMN].astype(str).to_numpy()

print('Number of classes :', len(label_encoder.classes_))
print('Number of features:', len(feature_cols_final))
print('Training NaN cells:', int(X.isna().sum().sum()))
print('Test NaN cells    :', int(X_test.isna().sum().sum()))
print('Classes:')
print(list(label_encoder.classes_))

Number of classes : 13
Number of features: 106
Training NaN cells: 0
Test NaN cells    : 0
Classes:
['Diya_Na', 'Gammiris', 'Ingini', 'Iriveriya', 'Kapparawalliya', 'Kora_Kaha', 'Kuringchan', 'Kurundu', 'Masbadda', 'Na', 'Rathu_Koboleela', 'Sudu_Koboleela', 'Wali_Kaha']


## 7. Create the training/validation split

The held-out test set is not used here. When `GROUP_COLUMN` is configured, all samples in a group remain together.

In [ ]:
if groups_all is None:
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_encoded,
        test_size=VAL_SIZE,
        stratify=y_encoded,
        random_state=RANDOM_STATE,
    )
    groups_train = None
    groups_val = None
else:
    n_splits_for_holdout = max(2, round(1 / VAL_SIZE))
    holdout_splitter = StratifiedGroupKFold(
        n_splits=n_splits_for_holdout,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    train_idx, val_idx = next(holdout_splitter.split(X, y_encoded, groups_all))
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    groups_train, groups_val = groups_all[train_idx], groups_all[val_idx]

print('Training rows  :', len(X_train))
print('Validation rows:', len(X_val))
print('Test rows      :', len(X_test))

train_class_counts = pd.Series(y_train).value_counts().sort_index()
val_class_counts = pd.Series(y_val).value_counts().sort_index()
if len(train_class_counts) != len(label_encoder.classes_) or len(val_class_counts) != len(label_encoder.classes_):
    raise ValueError('At least one class is absent from the train or validation split.')

print('✅ Every class is represented in train and validation data')

Training rows  : 5596
Validation rows: 1399
Test rows      : 130
✅ Every class is represented in train and validation data


## 8. Define models and leakage-safe pipelines

Scaling is applied only to SVM, Logistic Regression, and KNN. Tree models use median imputation but do not need scaling.

In [ ]:
def make_pipeline(classifier, needs_scaling):
    scaler_step = StandardScaler() if needs_scaling else 'passthrough'
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', scaler_step),
        ('classifier', classifier),
    ])


model_pipelines = {
    # Fast models first; Gradient Boosting intentionally runs last.
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=4000, random_state=RANDOM_STATE),
        needs_scaling=True,
    ),
    'K-Nearest Neighbors': make_pipeline(
        KNeighborsClassifier(),
        needs_scaling=True,
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        needs_scaling=False,
    ),
    'XGBoost': make_pipeline(
        XGBClassifier(
            objective='multi:softmax',
            num_class=len(label_encoder.classes_),
            eval_metric='mlogloss',
            tree_method='hist',
            random_state=RANDOM_STATE,
            n_jobs=1,             # outer search owns parallelism
            verbosity=0,
        ),
        needs_scaling=False,
    ),
    'SVM (RBF)': make_pipeline(
        SVC(
            kernel='rbf',
            probability=False,   # much faster; probabilities were not used
            cache_size=2048,
            random_state=RANDOM_STATE,
        ),
        needs_scaling=True,
    ),
    'Gradient Boosting': make_pipeline(
        GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            n_iter_no_change=10, # early stopping
            validation_fraction=0.10,
            tol=1e-4,
        ),
        needs_scaling=False,
    ),
}

if RUN_ONLY_MODELS is not None:
    unknown = sorted(set(RUN_ONLY_MODELS) - set(model_pipelines))
    if unknown:
        raise ValueError(f'Unknown RUN_ONLY_MODELS entries: {unknown}')
    model_pipelines = {name: model_pipelines[name] for name in RUN_ONLY_MODELS}

print('Models scheduled:')
for name in model_pipelines:
    print(' -', name)

Models scheduled:
 - Logistic Regression
 - K-Nearest Neighbors
 - Random Forest
 - XGBoost
 - SVM (RBF)
 - Gradient Boosting


## 9. Hyperparameter search spaces

The Gradient Boosting search is deliberately restricted because multiclass Gradient Boosting creates one regression tree per class at every boosting stage. With 13 species, the old 27-combination × 5-fold grid could build tens of thousands of trees.

In [ ]:
param_distributions = {
    'Logistic Regression': {
        'classifier__C': np.logspace(-3, 2, 20),
        'classifier__class_weight': [None, 'balanced'],
        'classifier__solver': ['lbfgs'],
    },
    'K-Nearest Neighbors': {
        'classifier__n_neighbors': list(range(3, 24, 2)),
        'classifier__weights': ['uniform', 'distance'],
        'classifier__p': [1, 2],
        'classifier__leaf_size': [20, 30, 40, 50],
    },
    'Random Forest': {
        'classifier__n_estimators': [200, 300, 500, 700],
        'classifier__max_depth': [None, 12, 20, 30, 40],
        'classifier__min_samples_split': [2, 4, 6, 10],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__max_features': ['sqrt', 'log2', 0.5, None],
        'classifier__class_weight': [None, 'balanced_subsample'],
    },
    'XGBoost': {
        'classifier__n_estimators': [150, 250, 400, 600],
        'classifier__max_depth': [3, 4, 5, 7],
        'classifier__learning_rate': [0.02, 0.04, 0.06, 0.10, 0.15],
        'classifier__subsample': [0.70, 0.85, 1.0],
        'classifier__colsample_bytree': [0.65, 0.80, 1.0],
        'classifier__min_child_weight': [1, 3, 5, 8],
        'classifier__reg_alpha': [0.0, 0.01, 0.1, 0.5],
        'classifier__reg_lambda': [0.5, 1.0, 2.0, 5.0],
    },
    'SVM (RBF)': {
        'classifier__C': np.logspace(-1, 2.5, 18),
        'classifier__gamma': ['scale', 'auto', 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
        'classifier__class_weight': [None, 'balanced'],
    },
    'Gradient Boosting': {
        # Small, high-value search space plus early stopping.
        'classifier__n_estimators': [60, 90, 120, 150],
        'classifier__learning_rate': [0.03, 0.05, 0.08, 0.10],
        'classifier__max_depth': [1, 2, 3],
        'classifier__min_samples_leaf': [1, 2, 4, 8],
        'classifier__subsample': [0.70, 0.85, 1.0],
        'classifier__max_features': [None, 'sqrt', 0.70],
    },
}

if groups_train is None:
    tuning_cv = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
else:
    tuning_cv = StratifiedGroupKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

print('Search iterations per model:')
for name in model_pipelines:
    print(f' - {name:<22}: {SEARCH_ITERATIONS[name]} iterations × {CV_SPLITS} folds')

Search iterations per model:
 - Logistic Regression   : 8 iterations × 3 folds
 - K-Nearest Neighbors   : 10 iterations × 3 folds
 - Random Forest         : 12 iterations × 3 folds
 - XGBoost               : 14 iterations × 3 folds
 - SVM (RBF)             : 10 iterations × 3 folds
 - Gradient Boosting     : 8 iterations × 3 folds


## 10. Helper functions for metrics, checkpoints, and reports

In [ ]:
def slugify(value):
    return re.sub(r'[^a-z0-9]+', '_', value.lower()).strip('_')


def metric_values(y_true, y_pred):
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }


def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return value


def feature_signature(columns):
    payload = '|'.join(columns).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


RUN_SIGNATURE = {
    'train_rows': int(len(X_train)),
    'validation_rows': int(len(X_val)),
    'feature_count': int(len(feature_cols_final)),
    'feature_hash': feature_signature(feature_cols_final),
    'classes': list(map(str, label_encoder.classes_)),
    'profile': TUNING_PROFILE,
    'cv_splits': int(CV_SPLITS),
    'selection_metric': SELECTION_METRIC,
    'random_state': int(RANDOM_STATE),
    'group_column': GROUP_COLUMN,
}


def checkpoint_file(model_name):
    return os.path.join(CHECKPOINT_PATH, f'{slugify(model_name)}_checkpoint.joblib')


def checkpoint_matches(checkpoint):
    return checkpoint.get('run_signature') == RUN_SIGNATURE


def save_confusion_matrix(y_true, y_pred, model_name, split_name, output_file):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
    )
    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=label_encoder.classes_,
    )
    fig, ax = plt.subplots(figsize=(10, 9))
    display.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
    ax.set_title(f'{split_name} Confusion Matrix — {model_name}')
    fig.tight_layout()
    fig.savefig(output_file, dpi=140, bbox_inches='tight')
    plt.close(fig)


def save_classification_report(y_true, y_pred, model_name, split_name):
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    output = os.path.join(
        REPORTS_PATH,
        f'{slugify(model_name)}_{split_name.lower()}_classification_report.csv',
    )
    report_df.to_csv(output)
    return report_df


print('✅ Helpers ready')

✅ Helpers ready


## 11. Tune every model and evaluate on validation data

Completed model searches are immediately checkpointed. If the runtime stops during Gradient Boosting, rerun this cell: the previously completed models will load from checkpoints.

In [20]:
tuned_models = {}
validation_rows = []

print('🔧 Tuning every configured model')
print('=' * 90)

for model_name, pipeline in model_pipelines.items():
    print(f'\n▶ {model_name}')
    path = checkpoint_file(model_name)
    checkpoint = None

    should_force = model_name in FORCE_RETRAIN_MODELS
    if RESUME_FROM_CHECKPOINTS and os.path.exists(path) and not should_force:
        loaded = joblib.load(path)
        if checkpoint_matches(loaded):
            checkpoint = loaded
            print('   ♻️ Loaded matching checkpoint')
        else:
            print('   ⚠️ Ignored stale checkpoint because data/configuration changed')

    if checkpoint is None:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[model_name],
            n_iter=SEARCH_ITERATIONS[model_name],
            scoring=SELECTION_METRIC,
            n_jobs=SEARCH_N_JOBS,
            cv=tuning_cv,
            refit=True,
            random_state=RANDOM_STATE,
            verbose=1,
            return_train_score=True,
            error_score='raise',
        )

        started = time.perf_counter()
        fit_kwargs = {}
        if groups_train is not None:
            fit_kwargs['groups'] = groups_train
        search.fit(X_train, y_train, **fit_kwargs)
        search_seconds = time.perf_counter() - started

        cv_results_df = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
        cv_results_file = os.path.join(
            REPORTS_PATH,
            f'{slugify(model_name)}_cv_results.csv',
        )
        cv_results_df.to_csv(cv_results_file, index=False)

        checkpoint = {
            'model_name': model_name,
            'best_estimator': search.best_estimator_,
            'best_params': search.best_params_,
            'best_cv_score': float(search.best_score_),
            'best_cv_std': float(
                cv_results_df.loc[cv_results_df['rank_test_score'] == 1, 'std_test_score'].iloc[0]
            ),
            'search_seconds': float(search_seconds),
            'n_candidates': int(len(cv_results_df)),
            'run_signature': RUN_SIGNATURE,
        }
        joblib.dump(checkpoint, path)
        print('   💾 Checkpoint saved')

    best_pipeline = checkpoint['best_estimator']

    validation_started = time.perf_counter()
    val_pred = best_pipeline.predict(X_val)
    validation_seconds = time.perf_counter() - validation_started
    val_metrics = metric_values(y_val, val_pred)

    tuned_models[model_name] = {
        **checkpoint,
        'validation_predictions': val_pred,
        'validation_metrics': val_metrics,
    }

    row = {
        'model': model_name,
        'best_cv_macro_f1': checkpoint['best_cv_score'],
        'cv_std': checkpoint['best_cv_std'],
        'validation_accuracy': val_metrics['accuracy'],
        'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
        'validation_macro_precision': val_metrics['macro_precision'],
        'validation_macro_recall': val_metrics['macro_recall'],
        'validation_macro_f1': val_metrics['macro_f1'],
        'validation_weighted_f1': val_metrics['weighted_f1'],
        'search_seconds': checkpoint['search_seconds'],
        'validation_predict_seconds': validation_seconds,
        'n_candidates': checkpoint['n_candidates'],
        'best_params': json.dumps(checkpoint['best_params'], default=json_safe),
    }
    validation_rows.append(row)

    print(f"   Best CV macro F1    : {checkpoint['best_cv_score']:.4f}")
    print(f"   Validation accuracy : {val_metrics['accuracy']:.4f}")
    print(f"   Validation macro F1 : {val_metrics['macro_f1']:.4f}")
    print(f"   Search time         : {checkpoint['search_seconds'] / 60:.2f} min")
    print(f"   Best parameters     : {checkpoint['best_params']}")

validation_comparison_df = pd.DataFrame(validation_rows).sort_values(
    ['best_cv_macro_f1', 'validation_macro_f1'],
    ascending=False,
).reset_index(drop=True)

validation_comparison_file = os.path.join(REPORTS_PATH, 'all_models_validation_comparison.csv')
validation_comparison_df.to_csv(validation_comparison_file, index=False)

print('\n' + '=' * 90)
print('📊 TUNED MODEL COMPARISON — TEST SET HAS NOT BEEN USED FOR SELECTION')
print('=' * 90)
display_columns = [
    'model',
    'best_cv_macro_f1',
    'validation_accuracy',
    'validation_macro_f1',
    'search_seconds',
]
print(validation_comparison_df[display_columns].to_string(index=False))

🔧 Tuning every configured model

▶ Logistic Regression
Fitting 3 folds for each of 8 candidates, totalling 24 fits
   💾 Checkpoint saved
   Best CV macro F1    : 0.9930
   Validation accuracy : 0.9936
   Validation macro F1 : 0.9936
   Search time         : 0.40 min
   Best parameters     : {'classifier__solver': 'lbfgs', 'classifier__class_weight': None, 'classifier__C': np.float64(2.636650898730358)}

▶ K-Nearest Neighbors
Fitting 3 folds for each of 10 candidates, totalling 30 fits
   💾 Checkpoint saved
   Best CV macro F1    : 0.9739
   Validation accuracy : 0.9786
   Validation macro F1 : 0.9785
   Search time         : 0.57 min
   Best parameters     : {'classifier__weights': 'distance', 'classifier__p': 1, 'classifier__n_neighbors': 3, 'classifier__leaf_size': 30}

▶ Random Forest
Fitting 3 folds for each of 12 candidates, totalling 36 fits
   💾 Checkpoint saved
   Best CV macro F1    : 0.9872
   Validation accuracy : 0.9850
   Validation macro F1 : 0.9849
   Search time        

c:\Users\User\Documents\Projects\Veda_Vision\ayurveda-recognition\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


   💾 Checkpoint saved
   Best CV macro F1    : 0.9982
   Validation accuracy : 0.9993
   Validation macro F1 : 0.9993
   Search time         : 1.75 min
   Best parameters     : {'classifier__gamma': 0.001, 'classifier__class_weight': 'balanced', 'classifier__C': np.float64(196.84194472866113)}

▶ Gradient Boosting
Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

## 12. Select the deployment model before test evaluation

The selected model is determined by tuned cross-validation macro F1. Validation macro F1 is only used as a tie-breaker. Test accuracy is not used for selection.

In [21]:
selected_model_name = validation_comparison_df.iloc[0]['model']
selected_best_params = tuned_models[selected_model_name]['best_params']

print('🏆 Selected before test evaluation:', selected_model_name)
print('Selection metric                : tuned CV macro F1')
print('Best parameters                :', selected_best_params)

plot_df = validation_comparison_df.copy()
x = np.arange(len(plot_df))
width = 0.26

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, plot_df['best_cv_macro_f1'], width, label='Best CV macro F1')
ax.bar(x, plot_df['validation_macro_f1'], width, label='Validation macro F1')
ax.bar(x + width, plot_df['validation_accuracy'], width, label='Validation accuracy')
ax.set_xticks(x, plot_df['model'], rotation=20)
ax.set_ylim(max(0, plot_df[['best_cv_macro_f1', 'validation_macro_f1', 'validation_accuracy']].min().min() - 0.05), 1.01)
ax.set_ylabel('Score')
ax.set_title('Tuned Cross-Validation and Validation Performance')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(LOGS_PATH, '05_tuned_validation_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()

NameError: name 'validation_comparison_df' is not defined

## 13. Refit every tuned model on all training rows and evaluate every model on test data

Each model keeps its own tuned hyperparameters. The full training data now includes the previous validation partition. Test results are reported for comparison, but they do not change the already selected deployment model.

In [ ]:
final_models = {}
test_rows = []
all_test_predictions = test_with_labels[['filename', 'view', 'true_label']].copy()

print('🧪 Refit and test-evaluate every tuned model')
print('=' * 90)

for model_name in validation_comparison_df['model']:
    print(f'\n▶ {model_name}')

    final_pipeline = clone(tuned_models[model_name]['best_estimator'])

    fit_started = time.perf_counter()
    final_pipeline.fit(X, y_encoded)
    full_refit_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    test_pred_encoded = final_pipeline.predict(X_test)
    test_predict_seconds = time.perf_counter() - predict_started

    test_metrics = metric_values(y_test_encoded, test_pred_encoded)
    test_pred_labels = label_encoder.inverse_transform(test_pred_encoded)

    model_slug = slugify(model_name)
    model_file = os.path.join(ALL_MODELS_PATH, f'{model_slug}_pipeline.pkl')
    joblib.dump(final_pipeline, model_file)

    predictions_df = test_with_labels[['filename', 'view', 'true_label']].copy()
    predictions_df['predicted_species'] = test_pred_labels
    predictions_df['correct'] = predictions_df['true_label'] == predictions_df['predicted_species']
    predictions_file = os.path.join(REPORTS_PATH, f'{model_slug}_test_predictions.csv')
    predictions_df.to_csv(predictions_file, index=False)

    all_test_predictions[f'{model_slug}_prediction'] = test_pred_labels

    report_df = save_classification_report(
        y_test_encoded,
        test_pred_encoded,
        model_name,
        'test',
    )
    save_confusion_matrix(
        y_test_encoded,
        test_pred_encoded,
        model_name,
        'Test',
        os.path.join(REPORTS_PATH, f'{model_slug}_test_confusion_matrix.png'),
    )

    classifier = final_pipeline.named_steps['classifier']
    effective_estimators = getattr(classifier, 'n_estimators_', None)

    final_models[model_name] = {
        'pipeline': final_pipeline,
        'test_predictions_encoded': test_pred_encoded,
        'test_predictions_labels': test_pred_labels,
        'test_metrics': test_metrics,
        'predictions_df': predictions_df,
        'classification_report_df': report_df,
        'model_file': model_file,
    }

    row = {
        'model': model_name,
        'selected_before_test': model_name == selected_model_name,
        'best_cv_macro_f1': tuned_models[model_name]['best_cv_score'],
        'validation_accuracy': tuned_models[model_name]['validation_metrics']['accuracy'],
        'validation_macro_f1': tuned_models[model_name]['validation_metrics']['macro_f1'],
        'test_accuracy': test_metrics['accuracy'],
        'test_balanced_accuracy': test_metrics['balanced_accuracy'],
        'test_macro_precision': test_metrics['macro_precision'],
        'test_macro_recall': test_metrics['macro_recall'],
        'test_macro_f1': test_metrics['macro_f1'],
        'test_weighted_f1': test_metrics['weighted_f1'],
        'search_seconds': tuned_models[model_name]['search_seconds'],
        'full_refit_seconds': full_refit_seconds,
        'test_predict_seconds': test_predict_seconds,
        'effective_estimators': effective_estimators,
        'best_params': json.dumps(tuned_models[model_name]['best_params'], default=json_safe),
        'saved_pipeline': model_file,
    }
    test_rows.append(row)

    print(f"   Test accuracy       : {test_metrics['accuracy']:.4f}")
    print(f"   Test balanced acc.  : {test_metrics['balanced_accuracy']:.4f}")
    print(f"   Test macro F1       : {test_metrics['macro_f1']:.4f}")
    print(f"   Full refit time     : {full_refit_seconds / 60:.2f} min")
    if effective_estimators is not None:
        print(f'   Effective estimators: {effective_estimators}')
    print(f'   Saved               : {model_file}')

final_comparison_df = pd.DataFrame(test_rows)
# Keep the pre-test selected model visually first, then order the rest by test macro F1.
final_comparison_df = final_comparison_df.sort_values(
    ['selected_before_test', 'test_macro_f1'],
    ascending=[False, False],
).reset_index(drop=True)

final_comparison_file = os.path.join(REPORTS_PATH, 'all_models_final_comparison.csv')
final_comparison_df.to_csv(final_comparison_file, index=False)
all_test_predictions.to_csv(
    os.path.join(REPORTS_PATH, 'all_models_test_predictions.csv'),
    index=False,
)

print('\n' + '=' * 90)
print('📊 FINAL TEST EVALUATION — EVERY MODEL')
print('=' * 90)
final_display_columns = [
    'model',
    'selected_before_test',
    'best_cv_macro_f1',
    'validation_macro_f1',
    'test_accuracy',
    'test_balanced_accuracy',
    'test_macro_f1',
    'full_refit_seconds',
]
print(final_comparison_df[final_display_columns].to_string(index=False))

## 14. Plot final comparison

In [ ]:
plot_df = final_comparison_df.sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
x = np.arange(len(plot_df))
width = 0.24

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - width, plot_df['best_cv_macro_f1'], width, label='CV macro F1')
ax.bar(x, plot_df['test_macro_f1'], width, label='Test macro F1')
ax.bar(x + width, plot_df['test_accuracy'], width, label='Test accuracy')
ax.set_xticks(x, plot_df['model'], rotation=20)
ax.set_ylim(max(0, plot_df[['best_cv_macro_f1', 'test_macro_f1', 'test_accuracy']].min().min() - 0.05), 1.01)
ax.set_ylabel('Score')
ax.set_title('All Tuned Models — Cross-Validation and Test Performance')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(LOGS_PATH, '05_all_tuned_models_test_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()

runtime_df = final_comparison_df.sort_values('search_seconds', ascending=True)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(runtime_df['model'], runtime_df['search_seconds'] / 60)
ax.set_xlabel('Hyperparameter search time (minutes)')
ax.set_title('Tuning Runtime by Model')
fig.tight_layout()
fig.savefig(os.path.join(LOGS_PATH, '05_tuning_runtime_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()

## 15. Detailed results for the model selected before test evaluation

In [ ]:
selected_result = final_models[selected_model_name]
selected_predictions = selected_result['predictions_df']
selected_metrics = selected_result['test_metrics']

print('🏆 Selected model:', selected_model_name)
print('Best parameters :', selected_best_params)
print()
print('TEST METRICS')
for metric_name, value in selected_metrics.items():
    print(f'{metric_name:<20}: {value:.4f}')

print('\nCLASSIFICATION REPORT')
print(selected_result['classification_report_df'].to_string())

print('\nPER-SPECIES ACCURACY')
per_species = selected_predictions.groupby('true_label')['correct'].agg(['mean', 'count'])
per_species.columns = ['accuracy', 'n_samples']
print(per_species.sort_values('accuracy').to_string())
per_species.to_csv(os.path.join(REPORTS_PATH, 'selected_model_per_species_accuracy.csv'))

print('\nPER-VIEW ACCURACY')
per_view = selected_predictions.groupby('view')['correct'].agg(['mean', 'count'])
per_view.columns = ['accuracy', 'n_samples']
print(per_view.to_string())
per_view.to_csv(os.path.join(REPORTS_PATH, 'selected_model_per_view_accuracy.csv'))

print('\nMOST COMMON MISCLASSIFICATIONS')
mistakes = selected_predictions[~selected_predictions['correct']]
if mistakes.empty:
    print('No test misclassifications 🎉')
else:
    confusion_pairs = (
        mistakes.groupby(['true_label', 'predicted_species'])
        .size()
        .sort_values(ascending=False)
        .rename('count')
    )
    print(confusion_pairs.head(30).to_string())
    confusion_pairs.to_csv(os.path.join(REPORTS_PATH, 'selected_model_misclassification_pairs.csv'))

## 16. Feature importance / influence for the selected model

- Tree models: native feature importance.
- Logistic Regression: mean absolute coefficient.
- SVM or KNN: permutation importance on the validation set.

In [ ]:
selected_validation_pipeline = tuned_models[selected_model_name]['best_estimator']
selected_classifier = selected_validation_pipeline.named_steps['classifier']

if hasattr(selected_classifier, 'feature_importances_'):
    importance_values = selected_classifier.feature_importances_
    importance_method = 'Native feature importance'
elif hasattr(selected_classifier, 'coef_'):
    importance_values = np.mean(np.abs(selected_classifier.coef_), axis=0)
    importance_method = 'Mean absolute coefficient'
else:
    # Limit the sample size so permutation importance stays practical.
    sample_size = min(1200, len(X_val))
    sampled_positions = np.random.default_rng(RANDOM_STATE).choice(
        len(X_val), size=sample_size, replace=False
    )
    X_val_sample = X_val.iloc[sampled_positions]
    y_val_sample = np.asarray(y_val)[sampled_positions]
    result = permutation_importance(
        selected_validation_pipeline,
        X_val_sample,
        y_val_sample,
        scoring=SELECTION_METRIC,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=SEARCH_N_JOBS,
    )
    importance_values = result.importances_mean
    importance_method = 'Permutation importance'

importance_df = pd.DataFrame({
    'feature': feature_cols_final,
    'importance': importance_values,
}).sort_values('importance', ascending=False)

importance_df.to_csv(
    os.path.join(REPORTS_PATH, 'selected_model_feature_importance.csv'),
    index=False,
)

print(importance_method)
print(importance_df.head(20).to_string(index=False))

top_features = importance_df.head(20).sort_values('importance')
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_features['feature'], top_features['importance'])
ax.set_xlabel('Importance')
ax.set_title(f'Top 20 Features — {selected_model_name}\n{importance_method}')
fig.tight_layout()
fig.savefig(os.path.join(LOGS_PATH, '05_selected_model_feature_importance.png'), dpi=140, bbox_inches='tight')
plt.show()

## 17. Save selected deployment artifacts

The full pipeline is the recommended artifact because it contains imputation, optional scaling, and the classifier in the correct order.

In [ ]:
selected_pipeline = final_models[selected_model_name]['pipeline']
selected_classifier = selected_pipeline.named_steps['classifier']
selected_imputer = selected_pipeline.named_steps['imputer']
selected_scaler = selected_pipeline.named_steps['scaler']
selected_preprocessor = Pipeline([
    ('imputer', selected_imputer),
    ('scaler', selected_scaler),
])

# Recommended deployment artifact.
joblib.dump(selected_pipeline, os.path.join(MODEL_PATH, 'species_classifier_pipeline.pkl'))

# Separate components are saved for inspection/backward compatibility.
joblib.dump(selected_classifier, os.path.join(MODEL_PATH, 'species_classifier.pkl'))
joblib.dump(selected_preprocessor, os.path.join(MODEL_PATH, 'preprocessor.pkl'))
joblib.dump(selected_imputer, os.path.join(MODEL_PATH, 'imputer.pkl'))
joblib.dump(selected_scaler, os.path.join(MODEL_PATH, 'scaler.pkl'))
joblib.dump(label_encoder, os.path.join(MODEL_PATH, 'label_encoder.pkl'))
joblib.dump(feature_cols_final, os.path.join(MODEL_PATH, 'feature_columns.pkl'))

metadata = {
    'selected_model': selected_model_name,
    'selection_rule': 'highest tuned cross-validation macro F1 before test evaluation',
    'best_params': selected_best_params,
    'test_metrics': selected_metrics,
    'feature_columns': feature_cols_final,
    'classes': list(label_encoder.classes_),
    'tuning_profile': TUNING_PROFILE,
    'cv_splits': CV_SPLITS,
    'selection_metric': SELECTION_METRIC,
    'random_state': RANDOM_STATE,
    'group_column': GROUP_COLUMN,
    'recommended_artifact': 'species_classifier_pipeline.pkl',
}

with open(os.path.join(MODEL_PATH, 'model_metadata.json'), 'w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2, default=json_safe)

print('✅ Selected deployment artifacts saved')
print('Recommended:', os.path.join(MODEL_PATH, 'species_classifier_pipeline.pkl'))
print('All tuned final models:', ALL_MODELS_PATH)
print('All reports:', REPORTS_PATH)

## 18. Verify saved pipeline with a round-trip prediction test

In [ ]:
loaded_pipeline = joblib.load(os.path.join(MODEL_PATH, 'species_classifier_pipeline.pkl'))
loaded_encoder = joblib.load(os.path.join(MODEL_PATH, 'label_encoder.pkl'))
loaded_features = joblib.load(os.path.join(MODEL_PATH, 'feature_columns.pkl'))

sample_raw_features = X_test[loaded_features].head(5)
round_trip_encoded = loaded_pipeline.predict(sample_raw_features)
round_trip_labels = loaded_encoder.inverse_transform(round_trip_encoded)
expected_labels = final_models[selected_model_name]['test_predictions_labels'][:5]

assert np.array_equal(round_trip_labels, expected_labels), 'Saved pipeline predictions changed after loading.'
print('✅ Saved pipeline reload test passed')
print('Example predictions:', list(round_trip_labels))

## 19. Final summary

In [ ]:
selected_row = final_comparison_df[
    final_comparison_df['model'] == selected_model_name
].iloc[0]

print('📊 FINAL SUMMARY')
print('=' * 90)
print(final_comparison_df[[
    'model',
    'selected_before_test',
    'best_cv_macro_f1',
    'validation_macro_f1',
    'test_accuracy',
    'test_balanced_accuracy',
    'test_macro_f1',
    'search_seconds',
]].to_string(index=False))
print('=' * 90)
print('Selected model               :', selected_model_name)
print('Selected using               : tuned CV macro F1 before test evaluation')
print(f"Selected model test accuracy : {selected_row['test_accuracy']:.4f}")
print(f"Selected model test macro F1 : {selected_row['test_macro_f1']:.4f}")
print('Best parameters              :', selected_best_params)
print('Recommended saved pipeline   :', os.path.join(MODEL_PATH, 'species_classifier_pipeline.pkl'))
print('Comparison CSV               :', os.path.join(REPORTS_PATH, 'all_models_final_comparison.csv'))
print()
print('For a new extracted feature row:')
print('  1. Reorder it using feature_columns.pkl')
print('  2. Add view_top exactly as done in this notebook')
print('  3. Call species_classifier_pipeline.pkl.predict(raw_feature_dataframe)')
print('  4. Decode the numeric class with label_encoder.pkl')